# 文の意味的類似度 (STS) でモデルを評価する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/09_sentence_embeddings/sts_eval.ipynb)

NLP クックブック 09-06 ／ 文埋め込みとクラスタリング

文埋め込みモデルが出す類似度が、人間の判断とどれくらい合っているかを測りたい。

In [1]:
!pip install -q datasets

In [2]:
from datasets import load_dataset
from scipy.stats import pearsonr, spearmanr
from sentence_transformers import SentenceTransformer

ds = load_dataset("sentence-transformers/stsb", split="test")
print(ds[0])

for name in ["sentence-transformers/all-MiniLM-L6-v2", "intfloat/multilingual-e5-small"]:
    model = SentenceTransformer(name)
    prefix = "query: " if "e5" in name else ""
    e1 = model.encode([prefix + s for s in ds["sentence1"]], normalize_embeddings=True, batch_size=64)
    e2 = model.encode([prefix + s for s in ds["sentence2"]], normalize_embeddings=True, batch_size=64)
    cos = (e1 * e2).sum(axis=1)
    print(f"{name:42} Spearman = {spearmanr(cos, ds['score']).statistic:.4f}  Pearson = {pearsonr(cos, ds['score']).statistic:.4f}")

{'sentence1': 'A girl is styling her hair.', 'sentence2': 'A girl is brushing her hair.', 'score': 0.5}
sentence-transformers/all-MiniLM-L6-v2     Spearman = 0.8203  Pearson = 0.8274
intfloat/multilingual-e5-small             Spearman = 0.8411  Pearson = 0.8213


- STS の評価は Spearman の順位相関で報告するのが慣例。
- 日本語なら JGLUE の JSTS や JSICK を使う。複数タスクでまとめて評価するなら MTEB / JMTEB。